# Dataset overview

This notebook explores the structure of `Twin-2k-500` dataset and its two HuggingFace configurations. It also identifies the appropriate configuration for training and evaluating a behavior-prediction model.

In [1]:
%load_ext jupyter_black

In [2]:
import json
import re

import matplotlib.pyplot as plt
import pandas as pd
from datasets import Value, load_dataset

pd.set_option("display.max_colwidth", None)

## Load both dataset configurations

In [3]:
DATASET_ID = "LLM-Digital-Twin/Twin-2K-500"

In [4]:
full_persona = load_dataset(DATASET_ID, "full_persona", split="data").cast_column(
    "pid", Value("string")
)

wave_split = load_dataset(DATASET_ID, "wave_split", split="data").cast_column(
    "pid", Value("string")
)

In [5]:
overview = pd.DataFrame(
    {
        "configuration": ["full_persona", "wave_split"],
        "participants": [len(full_persona), len(wave_split)],
        "columns": [
            ", ".join(full_persona.column_names),
            ", ".join(wave_split.column_names),
        ],
    }
)
display(overview)

assert set(full_persona["pid"]) == set(wave_split["pid"])
print("Participant sets match.")

,configuration,participants,columns
0,full_persona,2058,"pid, persona_text, persona_summary, persona_json"
1,wave_split,2058,"pid, wave1_3_persona_text, wave1_3_persona_json, wave4_Q_wave1_3_A, wave4_Q_wave4_A"


Participant sets match.


In [6]:
print(wave_split["wave1_3_persona_text"][0])

Which part of the United States do you currently live in?
Question Type: Single Choice
Options:
  1 - Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME)
  2 - Midwest (ND, SD, NE, KS, MN, IA, MO, WI, IL, MI, IN, OH)
  3 - South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL)
  4 - West (WA, OR, ID, MT, WY, CA, NV, UT, CO, AZ, NM)
  5 - Pacific (HI, AK)
Answer: 3 - South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL)

What is the sex that you were assigned at birth?
Question Type: Single Choice
Options:
  1 - Male
  2 - Female
Answer: 1 - Male

How old are you?
Question Type: Single Choice
Options:
  1 - 18-29
  2 - 30-49
  3 - 50-64
  4 - 65+
Answer: 1 - 18-29

What is the highest level of schooling or degree that you have completed?
Question Type: Single Choice
Options:
  1 - Less than high school
  2 - High school graduate
  3 - Some college, no degree
  4 - Associate's degree
  5 - College graduate/some postgrad
  6 - Postgraduate
Answer: 3 - S

In [7]:
json.loads(wave_split["wave1_3_persona_json"][0])

[{'ElementType': 'Block',
  'BlockName': 'Demographics',
  'BlockType': 'Standard',
  'Questions': [{'QuestionID': 'QID11',
    'QuestionText': 'Which part of the United States do you currently live in?',
    'QuestionType': 'MC',
    'Options': ['Northeast (PA, NY, NJ, RI, CT, MA, VT, NH, ME)',
     'Midwest (ND, SD, NE, KS, MN, IA, MO, WI, IL, MI, IN, OH)',
     'South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL)',
     'West (WA, OR, ID, MT, WY, CA, NV, UT, CO, AZ, NM)',
     'Pacific (HI, AK)'],
    'Settings': {'Selector': 'SAVR',
     'SubSelector': 'TX',
     'ForceResponse': 'ON'},
    'Answers': {'SelectedByPosition': 3,
     'SelectedText': 'South (TX, OK, AR, LA, KY, TN, MS, AL, WV, DC, MD, DE, VA, NC, SC, GA, FL)'}},
   {'QuestionID': 'QID12',
    'QuestionText': 'What is the sex that you were assigned at birth?',
    'QuestionType': 'MC',
    'Options': ['Male', 'Female'],
    'Settings': {'Selector': 'SAVR',
     'SubSelector': 'TX',
     'ForceRes

In [8]:
def normalize_text(text: str | None) -> str:
    return re.sub(r"\s+", " ", text or "").strip()


def extract_questions(raw_json: str) -> set[tuple]:
    return {
        (
            normalize_text(block.get("BlockName")),
            question.get("QuestionID"),
            question.get("QuestionType"),
            normalize_text(question.get("QuestionText")),
        )
        for block in json.loads(raw_json)
        for question in block.get("Questions", [])
    }

In [9]:
full_by_pid = {row["pid"]: row for row in full_persona}
wave_by_pid = {row["pid"]: row for row in wave_split}

records = []
for pid, wave_row in wave_by_pid.items():
    full = extract_questions(full_by_pid[pid]["persona_json"])
    persona = extract_questions(wave_row["wave1_3_persona_json"])
    earlier = extract_questions(wave_row["wave4_Q_wave1_3_A"])
    wave4 = extract_questions(wave_row["wave4_Q_wave4_A"])

    records.append(
        {
            "pid": pid,
            "persona_missing_from_full": len(persona - full),
            "wave4_missing_from_full": len(wave4 - full),
            "holdout_overlap_with_persona": len(earlier & persona),
            "earlier_wave4_difference": len(earlier ^ wave4),
        }
    )

checks = pd.DataFrame(records)

In [10]:
check_summary = pd.Series(
    {
        "persona_subset_of_full_pct": 100
        * checks["persona_missing_from_full"].eq(0).mean(),
        "wave4_subset_of_full_pct": 100
        * checks["wave4_missing_from_full"].eq(0).mean(),
        "holdout_disjoint_from_persona_pct": 100
        * checks["holdout_overlap_with_persona"].eq(0).mean(),
        "same_earlier_wave4_questions_pct": 100
        * checks["earlier_wave4_difference"].eq(0).mean(),
    }
)
display(check_summary)

persona_subset_of_full_pct           100.0
wave4_subset_of_full_pct             100.0
holdout_disjoint_from_persona_pct    100.0
same_earlier_wave4_questions_pct     100.0
dtype: float64

### Summary

`Twin-2K-500` provides two dataset configurations:
1. `full_persona` 
    - `persona_json`: contains participant questions and answers collected across waves 1-4.
2. `wave_split`
    - `wave1_3_persona_json` and `wave1_3_persona_text`: contain responses from waves 1-3. This dataset is used to construct the participant's persona.
    - `wave4_Q_wave1_3_A`: contains a held out set of questions and responses from waves 1-3. 
    - `wave4_Q_wave4_A`: contains the participant's wave 4 response to the same held out set of questions. The response from wave 4 is used to evaluate model performance. 

We perform validation checks to confirm that both wave 1-3 persona and wave 4 questions from `wave_split` exists in `full_persona`. We also verified that the waves 1-3 held out questions are exlcuded from the persona and align with questions repeated in wave 4.

The persona is treated as a static profile constructed from the observed wave 1-3 responses, rather than as a chronological representation of the participant's state before each reponse. 


## Conclusion
The `wave_split` configuration will be used for model training and evaluation as it seperates the persona (`wave1_3_persona_json`, `wave1_3_persona_text`) from the held-out questions (`wave4_Q_wave1_3_A`, `wave4_Q_wave4_A`). The responses from the held-out questions as used as evaluation targets (ground truth). On the other hand, as the `full_persona` configuration contains both persona and held-out questions for evaluation, it may introduce target leakage.
